# 04 Export GeoJSON and KML

In [ ]:
import geopandas as gpd
import pandas as pd
import simplekml
import os

## Load predictions

In [ ]:
POLYGON_FOLDER = "data/crown_polygons"
PRED_CSV = "outputs/predictions.csv"

df_pred = pd.read_csv(PRED_CSV)

## Attach predictions to polygons

In [ ]:
all_polys = []

for gj in os.listdir(POLYGON_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix = os.path.splitext(gj)[0]
    g = gpd.read_file(os.path.join(POLYGON_FOLDER, gj))

    g["image_name"] = g["id"].apply(
        lambda x: f"{prefix}_{int(x):03d}.tif"
    )

    all_polys.append(g)

gdf = pd.concat(all_polys)
gdf = gdf.merge(df_pred,on="image_name",how="left")

## Export GeoJSON

In [ ]:
gdf.to_file("outputs/species_labeled.geojson", driver="GeoJSON")

## Export KML

In [ ]:
gdf_wgs = gdf.to_crs(epsg=4326)

kml = simplekml.Kml()

for _,row in gdf_wgs.iterrows():

    if pd.isna(row["label"]):
        continue

    pol = kml.newpolygon(
        name=row["label"],
        outerboundaryis=list(row.geometry.exterior.coords)
    )

kml.save("outputs/species_map.kml")